# MEP Routing with topologic_fast

This notebook demonstrates MEP (Mechanical, Electrical, Plumbing) routing using topologic_fast.
It shows how to create a graph from node and edge data, merge the geometry, and find shortest paths.

**Adapted from topologicpy MEP notebook**

This program is free software under the GNU Affero General Public License.

Key concepts:
- Creating vertices and edges from coordinate data
- Building graphs from topological data
- Finding shortest paths for MEP routing
- Visualizing results with Plotly

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
import numpy as np

## Create Sample MEP Network Data

In a real application, this data would come from CSV files. Here we create sample data
representing an MEP routing network with nodes (connection points) and edges (pipe/duct segments).

In [ ]:
# Sample edge data: edge_id, start_x, start_y, start_z, end_x, end_y, end_z
edges_data = [
    ("E1", -1.0, 0.0, 0.0, 0.0, 0.0, 0.0),
    ("E2", 0.0, 0.0, 0.0, 0.5, 0.0, 0.0),
    ("E3", 0.5, 0.0, 0.0, 1.0, 0.0, 0.0),
    ("E4", 0.0, 0.0, 0.0, 0.0, 0.5, 0.0),
    ("E5", 0.0, 0.5, 0.0, 0.0, 0.5, 0.5),
    ("E6", 0.0, 0.0, 0.0, 0.0, 0.0, 0.5),
    ("E7", 0.0, 0.0, 0.5, 0.0, -0.5, 0.5),
    ("E8", 1.0, 0.0, 0.0, 1.0, 0.5, 0.0),
    ("E9", 1.0, 0.5, 0.0, 1.0, 0.5, -0.5),
]

# Sample node data: node_id, x, y, z, adjacent_edges
nodes_data = [
    ("N1", -1.0, -0.5, 0.5, ["E1"]),
    ("N2", 1.0, 0.5, -0.5, ["E9"]),
    ("N3", 0.0, 0.5, 0.5, ["E5"]),
    ("N4", 0.0, -0.5, 0.5, ["E7"]),
]

print(f"Created {len(edges_data)} edges and {len(nodes_data)} nodes")

## Create Topological Elements

Build vertices and edges from the coordinate data.

In [ ]:
# Store edges with their IDs
edges = []
edge_ids = {}

for edge_data in edges_data:
    edge_id, sx, sy, sz, ex, ey, ez = edge_data
    sv = tf.Vertex.ByCoordinates(sx, sy, sz)
    ev = tf.Vertex.ByCoordinates(ex, ey, ez)
    edge = tf.Edge.ByStartVertexEndVertex(sv, ev)
    edges.append(edge)
    edge_ids[edge_id] = edge
    
print(f"Created {len(edges)} edge objects")

In [ ]:
# Create vertices (nodes) and connect them to adjacent edges
vertices = []
new_edges = []  # Edges connecting nodes to the main network

for node_data in nodes_data:
    node_id, x, y, z, adj_edge_ids = node_data
    v = tf.Vertex.ByCoordinates(x, y, z)
    vertices.append(v)
    
    # Connect this node to its adjacent edges
    for adj_edge_id in adj_edge_ids:
        if adj_edge_id in edge_ids:
            edge = edge_ids[adj_edge_id]
            # Find nearest point on edge
            # NOTE: NearestVertex not yet implemented in topologic_fast
            # For now, connect to start vertex of the edge
            edge_start = edge.StartVertex()
            new_edge = tf.Edge.ByStartVertexEndVertex(edge_start, v)
            new_edges.append(new_edge)

print(f"Created {len(vertices)} node vertices and {len(new_edges)} connecting edges")

## Build the Graph

Combine all edges and create a graph for pathfinding.

In [ ]:
# Combine all edges
all_edges = edges + new_edges

# Collect all unique vertices from edges
all_vertices = []
for edge in all_edges:
    verts = edge.Vertices()
    all_vertices.extend(verts)

# Create graph from vertices and edges
graph = tf.Graph.ByVerticesEdges(all_vertices, all_edges)

print(f"Graph created with:")
print(f"  Vertices (nodes): {graph.Order()}")
print(f"  Edges (connections): {graph.Size()}")

## Find Shortest Path

Find the shortest path between two points in the MEP network.

In [ ]:
# Define start and end points
start_point = tf.Vertex.ByCoordinates(-1.0, -0.5, 0.5)
end_point = tf.Vertex.ByCoordinates(1.0, 0.5, -0.5)

# Find nearest vertices in the graph by computing distances manually
# (NearestVertex is not yet implemented in topologic_fast)
def find_nearest_vertex(graph, point):
    """Find the nearest vertex in a graph to a given point."""
    graph_vertices = graph.Vertices()
    point_coords = point.Coordinates()
    
    min_dist = float('inf')
    nearest = None
    
    for v in graph_vertices:
        v_coords = v.Coordinates()
        dist = ((v_coords[0] - point_coords[0])**2 + 
                (v_coords[1] - point_coords[1])**2 + 
                (v_coords[2] - point_coords[2])**2)**0.5
        if dist < min_dist:
            min_dist = dist
            nearest = v
    
    return nearest

v1 = find_nearest_vertex(graph, start_point)
v2 = find_nearest_vertex(graph, end_point)

print(f"Start vertex: {v1.Coordinates()}")
print(f"End vertex: {v2.Coordinates()}")

# Find shortest path
shortest_path = graph.Path(v1, v2)

if shortest_path:
    path_vertices = shortest_path.Vertices()
    path_edges = shortest_path.Edges()
    print(f"\nShortest path found:")
    print(f"  Path vertices: {len(path_vertices)}")
    print(f"  Path edges: {len(path_edges)}")
    print(f"  Graph distance: {graph.Distance(v1, v2)} steps")
else:
    print("No path found between the specified points")

## Visualize the MEP Network

Create a 3D visualization showing the network and the shortest path.

In [ ]:
def visualize_mep_network(graph, shortest_path=None):
    """Create a 3D visualization of the MEP network."""
    fig = go.Figure()
    
    # Plot all graph edges
    graph_edges = graph.Edges()
    for edge in graph_edges:
        verts = edge.Vertices()
        if len(verts) == 2:
            p1 = verts[0].Coordinates()
            p2 = verts[1].Coordinates()
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                z=[p1[2], p2[2]],
                mode='lines',
                line=dict(color='gray', width=3),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Plot all graph vertices
    graph_vertices = graph.Vertices()
    vertex_coords = [v.Coordinates() for v in graph_vertices]
    x = [c[0] for c in vertex_coords]
    y = [c[1] for c in vertex_coords]
    z = [c[2] for c in vertex_coords]
    
    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(size=5, color='blue'),
        name='Network Nodes',
        hoverinfo='text',
        hovertext=[f'({c[0]:.2f}, {c[1]:.2f}, {c[2]:.2f})' for c in vertex_coords]
    ))
    
    # Plot shortest path if provided
    if shortest_path:
        path_edges = shortest_path.Edges()
        for edge in path_edges:
            verts = edge.Vertices()
            if len(verts) == 2:
                p1 = verts[0].Coordinates()
                p2 = verts[1].Coordinates()
                fig.add_trace(go.Scatter3d(
                    x=[p1[0], p2[0]],
                    y=[p1[1], p2[1]],
                    z=[p1[2], p2[2]],
                    mode='lines',
                    line=dict(color='red', width=8),
                    showlegend=False,
                    hoverinfo='skip'
                ))
        
        # Mark path endpoints
        path_vertices = shortest_path.Vertices()
        if len(path_vertices) >= 2:
            start = path_vertices[0].Coordinates()
            end = path_vertices[-1].Coordinates()
            fig.add_trace(go.Scatter3d(
                x=[start[0], end[0]],
                y=[start[1], end[1]],
                z=[start[2], end[2]],
                mode='markers',
                marker=dict(size=10, color=['green', 'red']),
                name='Start/End Points'
            ))
    
    fig.update_layout(
        title='MEP Routing Network',
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            aspectmode='data'
        ),
        width=900,
        height=700
    )
    
    return fig

In [ ]:
# Create and display visualization
fig = visualize_mep_network(graph, shortest_path)
fig.show()

## Network Analysis

Analyze the graph properties of the MEP network.

In [ ]:
print("MEP Network Analysis")
print("=" * 40)
print(f"Number of junction points: {graph.Order()}")
print(f"Number of pipe/duct segments: {graph.Size()}")
print(f"Network density: {graph.Density():.3f}")
print(f"Network diameter: {graph.Diameter()} (max path length)")
print(f"Max connections at a junction: {graph.MaximumDelta()}")
print(f"Min connections at a junction: {graph.MinimumDelta()}")

# Check for isolated nodes (dead ends that might need attention)
isolated = graph.IsolatedVertices()
print(f"Isolated nodes (unconnected): {len(isolated)}")

## Junction Connectivity Analysis

In [ ]:
graph_vertices = graph.Vertices()

print("Junction Point Connectivity:")
print("-" * 50)

connectivity_data = []
for v in graph_vertices:
    coords = v.Coordinates()
    degree = graph.VertexDegree(v)
    connectivity_data.append((coords, degree))
    
# Sort by degree (most connected first)
connectivity_data.sort(key=lambda x: -x[1])

for coords, degree in connectivity_data:
    print(f"  Junction at ({coords[0]:6.2f}, {coords[1]:6.2f}, {coords[2]:6.2f}): {degree} connections")

## Summary

This notebook demonstrated:

1. **Creating MEP network topology** from coordinate data
2. **Building a graph** representing the pipe/duct network
3. **Finding shortest paths** for routing optimization
4. **Visualizing the network** in 3D with Plotly
5. **Analyzing network properties** like connectivity and diameter

### Not Yet Implemented in topologic_fast:
- `Vertex.NearestVertex()` - Finding nearest vertex on a topology
- `Topology.SetDictionary()` - Setting dictionary attributes on topologies
- `Dictionary` class - For storing attributes on topologies

### Applications:
- HVAC duct routing optimization
- Plumbing pipe layout
- Electrical conduit routing
- MEP clash detection coordination